# NuWro Fake Data Distributions

The NuWro fake-data twin of `open_data_distributions.ipynb`: the same set of distribution plots, with the
NuWro fake data sample playing the data role and the GENIE-based stacked prediction normalized to the NuWro
POT of each run period (`wc_net_weight_nuwro`; runs 1, 2, 3, 4a, 4c, 5, with the run 4b/4d overlays folded
into the run 4c group).  EXT, dirt, and real data have no weight in the NuWro config (the fake data contains
no beam-off or dirt component), so the prediction stacks are overlay MC only.  Every plot carries
reweighting (GENIE/flux/reint multisim) and detector-variation systematic uncertainty bands and is saved as
PNG and PDF under `plots/nuwro_fake_data_distributions/`, in the same subdirectory tree as the open-data
notebook:

```
plots/nuwro_fake_data_distributions/
├── previously_existing_selections/      # WC generic, NC Delta->Ngamma, inclusive 1gamma, inclusive nueCC
├── bdt_scores_all_topologies/           # 20-topology BDT probability grid over the full preselection
├── reco_Enu_all_topologies/             # 20-topology reco Enu grids (NuWro runs 1-5 combined, per-run)
│                                        #   plus individual per-topology reco Enu plots
├── one_bin_selections/                  # one-bin total-count plots for the 1gamma and nueCC selections
├── 1g_selections/<sel>/                 # 1gNp, 1g0p, 1gNp1mu, 1g0p1mu
├── 2g_selections/<sel>/                 # NC1pi0_Np/0p, numuCC1pi0_Np/0p, 1pi0_outFV, eta_other
├── nueCC_selections/<sel>/              # nueCC_Np, nueCC_0p
├── numuCC_selections/<sel>/             # numuCC_Np, numuCC_0p
├── NC0g_selection/NC_no_gamma/
└── other_selections/<sel>/              # multi_pi0, pi0_dalitz_decay
```

The selections, binnings, and truth breakdown (`del1g_detailed`) are identical to the open-data notebook.
All systematic covariances are cached under NuWro-specific selection names (`nwd_*`), separate from the
open-data notebook's cache entries.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['hatch.linewidth'] = 0.2
from matplotlib.patches import Rectangle
import numpy as np
import polars as pl
from tqdm.notebook import tqdm

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.file_locations import intermediate_files_location
from src.plot_helpers import make_histogram_plot, derive_normalizing_POT
from src.df_helpers import lazy_height, get_vals
from src.systematics import get_rw_sys_frac_cov_matrices, get_detvar_sys_frac_cov_matrices, get_pred_stat_cov
from src.signal_categories import train_category_labels, train_category_labels_latex
from src.signal_categories import (del1g_detailed_category_queries, del1g_detailed_category_labels,
                                   del1g_detailed_category_labels_latex, del1g_detailed_category_colors,
                                   del1g_detailed_category_hatches)

training = "all_vars_r15_2026_09_07"
reco_categories = train_category_labels
reco_category_labels_latex = train_category_labels_latex
prob_categories = ["prob_" + cat for cat in reco_categories]

# Independent systematics toggles, as in simple_generic_histogram.ipynb: reweighting
# (GENIE/flux/reint) multisim and detector-variation systematic bands on every plot.
# The covariance matrices are cached per selection+variable+binning, so only the first
# full run is slow (set dont_load_systematics_from_cache = True after the inputs change).
use_rw_systematics = True
use_detvar_systematics = True
dont_load_systematics_from_cache = False
num_bootstrap_rounds_detvar = 500
num_bootstrap_samples_detvar = 500

# With show_plots False the figures are still saved but not drawn inline (keeps the
# notebook file small when regenerating everything).
show_plots = True

plots_base = "nuwro_fake_data_distributions"

# File Loading

Same loading and BDT-prediction merging scheme as `open_data_multiclass_histograms.ipynb`: predictions only
exist for events passing the training preselection, only test events are used for the prediction, and their
weights are scaled up by the test fraction.  The NuWro fake data (never used in training) is kept whole and
plays the data role.

In [ ]:
print("loading all_df.parquet...")
all_df = pl.scan_parquet(f"{intermediate_files_location}/all_df.parquet")
print(f"num events in all_df: {lazy_height(all_df)}")

weights_df = None
detvar_presel_df = None
if use_rw_systematics:
    print("loading presel_weights_df.parquet (reweighting systematics)...")
    weights_df = pl.read_parquet(f"{intermediate_files_location}/presel_weights_df.parquet")
if use_detvar_systematics:
    print("loading detvar_presel_df_train_vars.parquet (detector-variation systematics)...")
    detvar_presel_df = pl.read_parquet(f"{intermediate_files_location}/detvar_presel_df_train_vars.parquet")

# this only includes predictions for events passing the preselection used during training
print("loading predictions.parquet...")
pred_df = pl.scan_parquet(f"../training_outputs/{training}/predictions.parquet")
print(f"num events in predictions.parquet: {lazy_height(pred_df)}")

print("merging all_df and predictions...")
merged_df_no_data_drop = all_df.join(
    pred_df,
    on=["filetype", "run", "subrun", "event"],
    how="left"
)

del all_df
del pred_df

In [ ]:
# The prediction is everything the NuWro weighting config gives a weight to: the overlay MC samples plus
# the reweighted NC coherent 1g / numuCC rad-corr samples.  Explicitly excluded here are the raw iso1g/del1g
# samples (their reweighted versions are kept), real data, EXT, and dirt (all three have no weight in the
# NuWro config -- the fake data contains no beam-off or dirt component), the fullosc sample (no NuWro
# weights), and the NuWro fake data itself, which plays the data role below.
full_pred = merged_df_no_data_drop.filter(
    ~pl.col("filetype").is_in(["data", "ext", "dirt_overlay",
                               "isotropic_one_gamma_overlay", "delete_one_gamma_overlay",
                               "nuwro_fake_data", "fullosc_overlay"])
)
full_data = merged_df_no_data_drop.filter(pl.col("filetype") == "nuwro_fake_data")
del merged_df_no_data_drop

for prob in prob_categories:
    full_pred = full_pred.with_columns(pl.col(prob).fill_null(-1))
    full_data = full_data.with_columns(pl.col(prob).fill_null(-1))

# generic neutrino preselection, used for everything below
generic_pred_df = full_pred.filter(pl.col("wc_kine_reco_Enu") > 0)
generic_data_df = full_data.filter(pl.col("wc_kine_reco_Enu") > 0)
del full_pred
del full_data

num_train_events = lazy_height(generic_pred_df.filter(pl.col("used_for_training") == True))
num_test_events = lazy_height(generic_pred_df.filter(pl.col("used_for_testing") == True))
print(f"num_train_events: {num_train_events}, num_test_events: {num_test_events}")
frac_test = num_test_events / (num_train_events + num_test_events)
print(f"weighting up preselected test prediction events by the fraction of test/train events: {frac_test:.3f}")

# only test events are used for the prediction (training events would be biased), so the NuWro weighting
# config gets scaled up by the test fraction; the NuWro fake data itself was never used in training and is
# kept whole
weight_vars = ["wc_net_weight_nuwro"]
generic_pred_df = generic_pred_df.with_columns([
    pl.when(pl.col("used_for_testing"))
    .then(pl.col(weight_var) / frac_test)
    .otherwise(pl.col(weight_var))
    .alias(weight_var)
    for weight_var in weight_vars
])

test_pred = generic_pred_df.filter(pl.col("used_for_testing") == True)
merged_df = pl.concat([test_pred, generic_data_df])
del generic_pred_df
del generic_data_df
del test_pred

In [ ]:
# Load only the columns actually used in this notebook (see open_data_multiclass_histograms.ipynb
# for why: collecting all ~1600 columns runs out of memory).  Intersected with the live schema so
# any columns that still require all_df.parquet to be reprocessed are skipped with a warning.
schema_names = merged_df.collect_schema().names()

needed_vars = [
    # bookkeeping / identity
    "filename", "filetype", "run", "subrun", "event",
    "used_for_training", "used_for_testing",
    # weights / normalization / run periods for the NuWro weighting config
    "wc_net_weight_nuwro", "normalizing_run_period_nuwro", "norm_goal_pot_nuwro",
    "detailed_run_period",
    # CV/spline weights needed by the reweighting systematic covariance machinery
    "wc_weight_cv", "wc_weight_spline",
    # signal-category breakdown used by make_histogram_plot
    "filetype_signal_category",
    # truth variables used by the del1g_detailed breakdown queries
    "normal_overlay", "del1g_overlay", "iso1g_overlay",
    "wc_truth_inFV", "wc_truth_NCDeltaRad", "wc_truth_numuCCDeltaRad",
    "wc_truth_0pi0", "wc_truth_1pi0", "wc_truth_multi_pi0",
    "wc_truth_Np", "wc_truth_0p", "wc_truth_0mu", "wc_truth_1mu",
    "wc_truth_isNC", "wc_truth_numuCC", "wc_truth_nueCC",
    "wc_truth_notnueCC", "wc_truth_notnumuCC",
    "wc_true_has_pi0_dalitz_decay", "wc_true_has_photonuclear_absorption",
    "true_num_gamma_pairconvert_in_FV", "true_num_gamma_pairconvert_in_FV_20_MeV",
    "wc_true_gamma_pairconversion_spacepoint_max_min_distance", "true_num_prim_gamma",
    # previously existing selection variables
    "wc_nc_delta_score", "wc_nue_score",
    "wc_shw_sp_n_20mev_showers", "wc_shw_sp_n_20br1_showers",
    "wc_single_photon_numu_score", "wc_single_photon_other_score",
    "wc_single_photon_ncpi0_score", "wc_single_photon_nue_score",
    # kinematic plotting variables
    "wc_kine_reco_Enu",
    "wc_reco_showerKE", "wc_reco_shower_theta", "wc_reco_shower_phi",
    "wc_reco_showerMomentum_0", "wc_reco_showerMomentum_1", "wc_reco_showerMomentum_2", "wc_reco_showerMomentum_3",
    "wc_reco_muonMomentum_0", "wc_reco_muonMomentum_1", "wc_reco_muonMomentum_2", "wc_reco_muonMomentum_3",
    "wc_reco_max_prim_proton_energy", "wc_reco_max_prim_proton_costheta", "wc_reco_max_prim_proton_phi",
    "wc_reco_sum_prim_proton_energy",
    "wc_reco_num_protons_35_MeV",
    "wc_kine_pio_energy_1", "wc_kine_pio_energy_2",
    "wc_kine_pio_theta_1", "wc_kine_pio_theta_2",
    "wc_kine_pio_phi_1", "wc_kine_pio_phi_2",
    "wc_kine_pio_angle", "wc_kine_pio_mass",
    # position in TPC
    "wc_reco_nuvtxX", "wc_reco_nuvtxY", "wc_reco_nuvtxZ",
    "wc_reco_distance_to_boundary", "wc_reco_nu_distance_to_boundary",
    "wc_reco_backwards_projected_dist",
]

load_vars = list(dict.fromkeys(needed_vars + prob_categories))  # de-dup, preserve order
missing_vars = [c for c in load_vars if c not in schema_names]
if missing_vars:
    print(f"WARNING: {len(missing_vars)} requested columns missing from all_df "
          f"(reprocess all_df.parquet to add them): {missing_vars}")
load_vars = [c for c in load_vars if c in schema_names]
print(f"collecting {len(load_vars)} columns...")

presel_merged_df = merged_df.select(load_vars).collect()
del merged_df
print(f"{presel_merged_df.height} preselected events loaded")

# Derived Kinematic Variables

A few plotting variables are not stored as scalars in `all_df` and are derived here (also applied to the
detvar sample below, so they get detector-variation bands too):

- `reco_muon_costheta` from the WC muon momentum 4-vector (`wc_reco_muonMomentum_*` is $(p_x, p_y, p_z, E)$ in GeV);
- `reco_leading_proton_KE` (MeV) and `reco_leading_proton_costheta`: null-masked aliases of the stored
  leading primary proton columns (KE in MeV and $p_z/|p|$ since the 2026-08-31 postprocessing update;
  this notebook requires dataframes processed with that update);
- the two-photon system (momentum, $\cos\theta$, summed energy) from the WC KINE $\pi^0$ variables (energies in MeV, angles in degrees);
- shower-proton, muon-proton, and muon-photon opening angles.  Note `wc_reco_max_prim_proton_phi` is `arctan2(px, py)` in degrees (see `postprocessing.py`), so $p_x \propto \sin\phi$ and $p_y \propto \cos\phi$ when reconstructing the direction.  The muon-photon opening angle is derived here rather than using `wc_muon_gamma_opening_angle`, which is a truth-level quantity that only exists for the `numuCC_rad_corrected` sample (with a $< 60^\circ$ cut from the radiative correction procedure) and is null for data and every other sample.

The summed primary proton KE (`wc_reco_sum_prim_proton_energy`, MeV) is stored directly by
postprocessing since the 2026-08-31 update.

In [ ]:
DEG = np.pi / 180.0

def add_derived_kinematics(df):
    """Adds the derived reco kinematic columns used for plotting (works on all_df and detvar dfs)."""

    # muon direction
    mu_p = (pl.col("wc_reco_muonMomentum_0")**2 + pl.col("wc_reco_muonMomentum_1")**2 + pl.col("wc_reco_muonMomentum_2")**2).sqrt()
    muon_valid = pl.col("wc_reco_muonMomentum_3") > 0

    # primary shower (photon / electron candidate) direction
    shw_p = (pl.col("wc_reco_showerMomentum_0")**2 + pl.col("wc_reco_showerMomentum_1")**2 + pl.col("wc_reco_showerMomentum_2")**2).sqrt()
    shower_valid = shw_p > 0

    # Leading primary proton.  Since the 2026-08-31 postprocessing update, the stored columns are
    # the leading primary proton KE in MeV and a true p_z/|p| costheta (sentinels -1 / -2 when
    # there is no reco primary proton); the aliases below just mask the sentinels to null.
    p_KE = pl.col("wc_reco_max_prim_proton_energy")
    proton_valid = p_KE > 0
    p_costh = pl.col("wc_reco_max_prim_proton_costheta").clip(-1, 1)
    p_sinth = (1 - p_costh**2).sqrt()
    p_phi = pl.col("wc_reco_max_prim_proton_phi") * DEG
    p_ux, p_uy, p_uz = p_sinth * p_phi.sin(), p_sinth * p_phi.cos(), p_costh

    # two-photon (WC KINE pi0 candidate) system
    e1, e2 = pl.col("wc_kine_pio_energy_1"), pl.col("wc_kine_pio_energy_2")
    pio_valid = (e1 > 0) & (e2 > 0)
    two_photon_mom = (e1**2 + e2**2 + 2 * e1 * e2 * (pl.col("wc_kine_pio_angle") * DEG).cos()).sqrt()
    two_photon_pz = e1 * (pl.col("wc_kine_pio_theta_1") * DEG).cos() + e2 * (pl.col("wc_kine_pio_theta_2") * DEG).cos()

    def opening_angle_deg(ax, ay, az, bx, by, bz):
        return (ax * bx + ay * by + az * bz).clip(-1, 1).arccos() / DEG

    return df.with_columns([
        pl.when(proton_valid).then(p_KE).otherwise(None).alias("reco_leading_proton_KE"),
        pl.when(proton_valid).then(p_costh).otherwise(None).alias("reco_leading_proton_costheta"),
        pl.when(muon_valid).then(pl.col("wc_reco_muonMomentum_2") / mu_p).otherwise(None).alias("reco_muon_costheta"),
        pl.when(pio_valid).then(two_photon_mom).otherwise(None).alias("reco_two_photon_momentum"),
        pl.when(pio_valid).then(two_photon_pz / two_photon_mom).otherwise(None).alias("reco_two_photon_costheta"),
        pl.when(pio_valid).then(e1 + e2).otherwise(None).alias("reco_two_photon_energy_sum"),
        pl.when(shower_valid & proton_valid).then(
            opening_angle_deg(pl.col("wc_reco_showerMomentum_0") / shw_p,
                              pl.col("wc_reco_showerMomentum_1") / shw_p,
                              pl.col("wc_reco_showerMomentum_2") / shw_p,
                              p_ux, p_uy, p_uz)
        ).otherwise(None).alias("reco_shower_proton_opening_angle"),
        pl.when(muon_valid & proton_valid).then(
            opening_angle_deg(pl.col("wc_reco_muonMomentum_0") / mu_p,
                              pl.col("wc_reco_muonMomentum_1") / mu_p,
                              pl.col("wc_reco_muonMomentum_2") / mu_p,
                              p_ux, p_uy, p_uz)
        ).otherwise(None).alias("reco_muon_proton_opening_angle"),
        # NOT wc_muon_gamma_opening_angle: that column is a truth-level quantity created only for
        # the numuCC_rad_corrected (delete-one-gamma) sample, with a < 60 deg cut from the
        # radiative correction procedure (see numuCC_rad_corr_1g_reweighting.py) -- it is null
        # for data and every other sample, so the reco opening angle is derived here instead
        pl.when(muon_valid & shower_valid).then(
            opening_angle_deg(pl.col("wc_reco_muonMomentum_0") / mu_p,
                              pl.col("wc_reco_muonMomentum_1") / mu_p,
                              pl.col("wc_reco_muonMomentum_2") / mu_p,
                              pl.col("wc_reco_showerMomentum_0") / shw_p,
                              pl.col("wc_reco_showerMomentum_1") / shw_p,
                              pl.col("wc_reco_showerMomentum_2") / shw_p)
        ).otherwise(None).alias("reco_muon_photon_opening_angle"),
    ])

derived_kinematic_columns = [
    "reco_leading_proton_KE", "reco_leading_proton_costheta",
    "reco_muon_costheta",
    "reco_two_photon_momentum", "reco_two_photon_costheta", "reco_two_photon_energy_sum",
    "reco_shower_proton_opening_angle", "reco_muon_proton_opening_angle",
    "reco_muon_photon_opening_angle",
]

presel_merged_df = add_derived_kinematics(presel_merged_df)

# BDT Reco Category Selections

Same cuts and orthogonality priorities as `open_data_multiclass_histograms.ipynb` (categories with no
tuned cut fall back to the argmax of the BDT probabilities).

In [ ]:
presel_merged_df = presel_merged_df.with_columns(
    pl.concat_list(prob_categories).list.arg_max().alias("reco_category_argmax_index")
)

reco_category_argmax_queries = []
for i, signal_category in enumerate(reco_categories):
    reco_category_argmax_queries.append(pl.col("reco_category_argmax_index") == i)

# tuned BDT score cut per category (None: no tuned cut, the category falls back to the argmax of the
# probabilities); the single source of truth for the selection cuts below and for the dashed cut lines
# drawn on the BDT score plots
bdt_cut_thresholds = {
    "1gNp": 0.3, "1g0p": 0.9, "1gNp1mu": 0.5, "1g0p1mu": 0.2, "1g_outFV": 0.5,
    "NC1pi0_Np": None, "NC1pi0_0p": None,
    "numuCC1pi0_Np": 0.1, "numuCC1pi0_0p": 0.15, "1pi0_outFV": 0.1,
    "nueCC_Np": 0.05, "nueCC_0p": 0.05,
    "numuCC_Np": 0.5, "numuCC_0p": 0.5,
    "other_outFV_dirt": None,
    "multi_pi0": 0.02, "eta_other": 0.01, "pi0_dalitz_decay": 0.1,
    "NC_no_gamma": None, "ext": None,
}

def _bdt_cut(name):
    thr = bdt_cut_thresholds[name]
    return None if thr is None else pl.col(f"prob_{name}") > thr

name_expr_priority_vals_w_None = [
    ("1gNp", _bdt_cut("1gNp"), 1),
    ("1g0p", _bdt_cut("1g0p"), 2),
    ("1gNp1mu", _bdt_cut("1gNp1mu"), 3),
    ("1g0p1mu", _bdt_cut("1g0p1mu"), 4),
    ("1g_outFV", _bdt_cut("1g_outFV"), 5),
    ("NC1pi0_Np", _bdt_cut("NC1pi0_Np"), 6),
    ("NC1pi0_0p", _bdt_cut("NC1pi0_0p"), 7),
    ("numuCC1pi0_Np", _bdt_cut("numuCC1pi0_Np"), 9),
    ("numuCC1pi0_0p", _bdt_cut("numuCC1pi0_0p"), 8), # 0p takes priority over Np in orthogonality
    ("1pi0_outFV", _bdt_cut("1pi0_outFV"), 10),
    ("nueCC_Np", _bdt_cut("nueCC_Np"), 12),
    ("nueCC_0p", _bdt_cut("nueCC_0p"), 11), # 0p takes priority over Np in orthogonality
    ("numuCC_Np", _bdt_cut("numuCC_Np"), 13),
    ("numuCC_0p", _bdt_cut("numuCC_0p"), 14),
    ("other_outFV_dirt", _bdt_cut("other_outFV_dirt"), 15),
    ("multi_pi0", _bdt_cut("multi_pi0"), 16),
    ("eta_other", _bdt_cut("eta_other"), 17),
    ("pi0_dalitz_decay", _bdt_cut("pi0_dalitz_decay"), 5.5), # high priority for dalitz, rare topology
    ("NC_no_gamma", _bdt_cut("NC_no_gamma"), 19),
    ("ext", _bdt_cut("ext"), 20),
]

name_expr_priority_vals_possible_overlap = []
for i, name_expr_priority_val_w_None in enumerate(name_expr_priority_vals_w_None):
    name, expr, priority = name_expr_priority_val_w_None
    if expr is None:
        expr = reco_category_argmax_queries[i]
    name_expr_priority_vals_possible_overlap.append((name, expr, priority))

name_expr_priority_vals_possible_overlap.sort(key=lambda x: x[2])

name_expr_priority_vals = []
for i, name_expr_priority_val_possible_overlap in enumerate(name_expr_priority_vals_possible_overlap):
    name, expr, priority = name_expr_priority_val_possible_overlap
    for j in range(i):
        expr = expr & ~name_expr_priority_vals_possible_overlap[j][1]
    name_expr_priority_vals.append((name, expr, priority))

reco_category_query_dic = {}
for name, expr, priority in name_expr_priority_vals:
    reco_category_query_dic[name] = expr

reco_category_queries = []
for reco_category in reco_categories:
    reco_category_queries.append(reco_category_query_dic[reco_category])

# Scoring The DetVar Sample With The BDT

The detvar sample has no stored BDT scores, so the same reco category cuts cannot be applied to it
directly.  Here it is scored with the trained BDT (in chunks, to bound memory), and the resulting
probability columns are cached to `intermediate_files` keyed by the training name; the cache is
validated against the current detvar dataframe's match keys before reuse.  The derived kinematics and
the argmax column are then added, so every reco category query and plotted variable also works on the
detvar sample.

In [ ]:
if detvar_presel_df is not None:
    detvar_match_keys = ["filetype", "detvar_sample", "vartype", "run", "subrun", "event"]
    detvar_prob_cache = f"{intermediate_files_location}/detvar_bdt_predictions_{training}.parquet"

    probs_df = None
    if os.path.exists(detvar_prob_cache) and not dont_load_systematics_from_cache:
        print("loading cached detvar BDT predictions...")
        probs_df = pl.read_parquet(detvar_prob_cache)
        if not probs_df.select(detvar_match_keys).equals(detvar_presel_df.select(detvar_match_keys)):
            print("cached detvar BDT predictions do not match the current detvar dataframe, recomputing...")
            probs_df = None

    if probs_df is None:
        import xgboost as xgb
        from src.ntuple_variables.variables import combined_training_vars

        print("scoring the detvar sample with the trained BDT (one-time, cached afterwards)...")
        model = xgb.XGBClassifier()
        model.load_model(f"../training_outputs/{training}/bdt.json")

        chunk_size = 200_000
        prob_chunks = []
        for start in tqdm(range(0, detvar_presel_df.height, chunk_size)):
            x = detvar_presel_df.slice(start, chunk_size).select(combined_training_vars).to_numpy().astype(np.float32)
            x[np.isinf(x)] = np.nan
            prob_chunks.append(model.predict_proba(x))
        all_probabilities = np.concatenate(prob_chunks)

        probs_df = detvar_presel_df.select(detvar_match_keys).with_columns([
            pl.Series(f"prob_{cat}", all_probabilities[:, i]) for i, cat in enumerate(reco_categories)
        ])
        probs_df.write_parquet(detvar_prob_cache)
        print(f"cached detvar BDT predictions to {detvar_prob_cache}")

    # row order of the cache was validated against the match keys above, so hstack the prob columns
    detvar_presel_df = pl.concat([detvar_presel_df, probs_df.select(prob_categories)], how="horizontal")
    del probs_df

    detvar_presel_df = add_derived_kinematics(detvar_presel_df)
    detvar_presel_df = detvar_presel_df.with_columns(
        pl.concat_list(prob_categories).list.arg_max().alias("reco_category_argmax_index")
    )

    # Slim the detvar dataframe to just the columns used downstream: the bookkeeping columns the
    # detvar covariance machinery needs, the BDT scores, and the plotted variables.  The full
    # ~1700-column, 3.9M-row detvar dataframe dominates the session's memory footprint, and
    # polars 1.34 has corrupted live in-memory data under heavy load in this project before
    # (see the streaming-race notes in save_PROfit_rootfiles.py), so a small footprint is not
    # just politeness -- it keeps the session out of the regime where that bug fires.
    detvar_keep = ["vartype", "detvar_sample", "wc_net_weight"]
    detvar_keep += [c for c in load_vars if c in detvar_presel_df.columns]
    detvar_keep += prob_categories + derived_kinematic_columns + ["reco_category_argmax_index"]
    detvar_presel_df = detvar_presel_df.select(list(dict.fromkeys(detvar_keep)))
    print(f"slimmed the detvar dataframe to {detvar_presel_df.width} columns")

# Plot Helpers

`plot_sel_var` makes one stacked NuWro-fake-data histogram (with the `del1g_detailed` breakdown) and saves
it as `plots/nuwro_fake_data_distributions/<subdir>/<name>.png/.pdf`.  Because `make_histogram_plot`'s
combined-dataframe path assumes `filetype == "data"` is the data, the prediction and the NuWro fake data
are passed separately.  Reweighting and detector-variation systematic bands are drawn following the
top-level toggles; each plot gets a NuWro-specific `selname` (`nwd_*`) so the systematic covariance cache
works per selection+variable, separately from the open-data notebook.  The detvar band is skipped (rw-only
band drawn) for any variable missing from the detvar sample.


In [ ]:
# Snapshot of the fully-built presel schema, so later cells fail fast with a clear message if
# the dataframe gets corrupted in memory mid-session (seen 2026-08-31 under heavy load with
# polars 1.34: the live schema lost half its columns and one column name became NUL bytes)
# instead of surfacing hours later as a confusing ColumnNotFoundError.
_expected_presel_columns = set(presel_merged_df.columns)

def assert_presel_df_intact():
    current = set(presel_merged_df.columns)
    if current != _expected_presel_columns:
        n_missing = len(_expected_presel_columns - current)
        missing = sorted(_expected_presel_columns - current)[:8]
        raise RuntimeError(
            f"presel_merged_df's schema changed in memory ({n_missing} columns missing, e.g. {missing}): "
            "in-session data corruption (known polars 1.34 issue under heavy load). "
            "Restart the kernel and rerun -- the systematics caches make the rerun fast."
        )


def sname(subdir, name):
    """savename under the organized output tree, creating the directory if needed."""
    os.makedirs(f"../plots/{plots_base}/{subdir}", exist_ok=True)
    return f"{plots_base}/{subdir}/{name}"


# forward-peaked cos(theta) distributions pile up under the default upper-right legend, so
# these variables get the legend on the upper left instead
left_legend_vars = {"reco_leading_proton_costheta", "reco_muon_costheta", "reco_two_photon_costheta"}


def plot_sel_var(sel_df, var, display_var, bins, subdir, name, title,
                 detvar_sel_df=None, selname=None, net_weight_var="wc_net_weight_nuwro", **kwargs):
    """One stacked NuWro-fake-data histogram, saved as png and pdf under plots/nuwro_fake_data_distributions/."""
    assert_presel_df_intact()
    savename = sname(subdir, name)
    # NuWro plays the data role, so the prediction and the fake data are passed separately
    # (make_histogram_plot's combined-df path takes filetype == "data" as the data)
    pred_sel_df = sel_df.filter((pl.col("filetype") != "nuwro_fake_data") & pl.col(net_weight_var).is_not_null())
    data_sel_df = sel_df.filter(pl.col("filetype") == "nuwro_fake_data")
    if detvar_sel_df is not None and var not in detvar_sel_df.columns:
        print(f"NOTE: {var} is not in the detvar sample, drawing the rw-only systematic band for {savename}")
        detvar_sel_df = None
    make_histogram_plot(
        pred_sel_df=pred_sel_df, data_sel_df=data_sel_df, bins=bins, var=var, display_var=display_var,
        net_weight_var=net_weight_var, breakdown_type="del1g_detailed",
        data_type="NuWro Fake Data",
        title=title, savename=savename, show=show_plots,
        # selname can be overridden to share cache entries with an identical selection/variable/binning made
        # elsewhere in this notebook (e.g. the BDT score grid panels)
        selname=selname if selname is not None else f"nwd_{subdir}_{name}".replace("/", "_"),
        use_rw_systematics=use_rw_systematics, weights_df=weights_df,
        use_detvar_systematics=use_rw_systematics and use_detvar_systematics and detvar_sel_df is not None,
        detvar_df=detvar_sel_df,
        num_bootstrap_rounds_detvar=num_bootstrap_rounds_detvar,
        num_bootstrap_samples_detvar=num_bootstrap_samples_detvar,
        dont_load_rw_from_systematic_cache=dont_load_systematics_from_cache,
        dont_load_detvar_from_systematic_cache=dont_load_systematics_from_cache,
        legend_loc="upper left" if var in left_legend_vars else "upper right",
        **kwargs,
    )
    if not show_plots:
        plt.close("all")


def plot_category_vars(reco_category, subdir, var_list):
    """All the (var, display_var, bins, name) plots for one BDT reco category selection."""
    i = reco_categories.index(reco_category)
    sel_df = presel_merged_df.filter(reco_category_queries[i])
    detvar_cat_df = detvar_presel_df.filter(reco_category_queries[i]) if detvar_presel_df is not None else None
    title = f"{reco_category_labels_latex[i]} Selection"
    for var, display_var, bins, name in var_list:
        plot_sel_var(sel_df, var, display_var, bins, f"{subdir}/{reco_category}", name, title,
                     detvar_sel_df=detvar_cat_df)

In [ ]:
# shared variable lists (var, display_var, bins, savename), reused across the sections below

photon_vars = [
    ("wc_reco_showerKE", r"Reco Shower KE (GeV)", np.linspace(0, 1.5, 16), "photon_energy"),
    ("wc_reco_shower_theta", r"Reco Shower $\theta$ (deg)", np.linspace(0, 180, 19), "photon_theta"),
    ("wc_reco_shower_phi", r"Reco Shower $\phi$ (deg)", np.linspace(-180, 180, 19), "photon_phi"),
]

leading_proton_vars = [
    ("reco_leading_proton_KE", "Reco Leading Proton KE (MeV)", np.linspace(0, 600, 21), "leading_proton_energy"),
    ("reco_leading_proton_costheta", r"Reco Leading Proton $\cos\theta$", np.linspace(-1, 1, 21), "leading_proton_costheta"),
]

proton_multiplicity_var = ("wc_reco_num_protons_35_MeV", "Reco Proton Multiplicity (> 35 MeV)", np.arange(-0.5, 7.5, 1.0), "proton_multiplicity")

summed_proton_vars = [
    ("wc_reco_sum_prim_proton_energy", "Reco Summed Proton KE (MeV)", np.linspace(0, 800, 21), "summed_proton_energy"),
]

muon_vars = [
    ("wc_reco_muonMomentum_3", "Reco Muon Energy (GeV)", np.linspace(0, 2.5, 21), "muon_energy"),
    ("reco_muon_costheta", r"Reco Muon $\cos\theta$", np.linspace(-1, 1, 21), "muon_costheta"),
]

position_vars = [
    ("wc_reco_nuvtxX", "Reco Neutrino Vertex X (cm)", np.linspace(0, 260, 27), "vtx_x"),
    ("wc_reco_nuvtxY", "Reco Neutrino Vertex Y (cm)", np.linspace(-120, 120, 25), "vtx_y"),
    ("wc_reco_nuvtxZ", "Reco Neutrino Vertex Z (cm)", np.linspace(0, 1040, 27), "vtx_z"),
    ("wc_reco_nu_distance_to_boundary", "Reco Neutrino Vertex Distance to Boundary (cm)", np.linspace(0, 130, 27), "nu_distance_to_boundary"),
    ("wc_reco_distance_to_boundary", "Reco Shower Vertex Distance to Boundary (cm)", np.linspace(0, 130, 27), "shower_distance_to_boundary"),
    ("wc_reco_backwards_projected_dist", "Reco Shower Backwards Projected Distance (cm)", np.linspace(-100, 2000, 43), "backwards_projected_dist"),
]

two_photon_leading_vars = [
    ("wc_kine_pio_energy_1", "Reco Leading Photon Energy (MeV)", np.linspace(0, 800, 21), "leading_photon_energy"),
    ("wc_kine_pio_energy_2", "Reco Subleading Photon Energy (MeV)", np.linspace(0, 400, 21), "subleading_photon_energy"),
    ("wc_kine_pio_theta_1", r"Reco Leading Photon $\theta$ (deg)", np.linspace(0, 180, 19), "leading_photon_theta"),
    ("wc_kine_pio_theta_2", r"Reco Subleading Photon $\theta$ (deg)", np.linspace(0, 180, 19), "subleading_photon_theta"),
]

def two_photon_system_vars(mass_bins):
    return [
        ("reco_two_photon_momentum", "Reco Two-Photon Momentum (MeV/c)", np.linspace(0, 1200, 25), "two_photon_momentum"),
        ("reco_two_photon_costheta", r"Reco Two-Photon $\cos\theta$", np.linspace(-1, 1, 21), "two_photon_costheta"),
        ("wc_kine_pio_angle", "Reco Two-Photon Opening Angle (deg)", np.linspace(0, 180, 19), "two_photon_opening_angle"),
        ("wc_kine_pio_mass", "Reco Two-Photon Invariant Mass (MeV)", mass_bins, "two_photon_mass"),
    ]

# Previously Existing Selections

The cut-based selections that existed before the multiclass BDT: WC generic neutrino, WC NC
$\Delta\rightarrow N\gamma$ ($Np$/$0p$), WC inclusive $1\gamma$ (Erin's), and WC inclusive $\nu_e$CC
($Np$/$0p$ and combined).  The detvar sample is filtered with the same cut expressions for the
detector-variation bands.

In [ ]:
erin_inclusive_1g_expr = (
    (pl.col("wc_shw_sp_n_20mev_showers") > 0)
    & (pl.col("wc_reco_nuvtxX") > 5.0) & (pl.col("wc_reco_nuvtxX") < 250.0)
    & (pl.col("wc_single_photon_numu_score") > 0.4)
    & (pl.col("wc_single_photon_other_score") > 0.2)
    & (pl.col("wc_single_photon_ncpi0_score") > -0.05)
    & (pl.col("wc_single_photon_nue_score") > -1.0)
    & (pl.col("wc_shw_sp_n_20br1_showers") == 1)
)

previously_existing_selections = [
    ("wc_generic", pl.col("wc_kine_reco_Enu") > 0,
     "WC Generic Neutrino Selection"),
    ("wc_nc_delta_Np", (pl.col("wc_nc_delta_score") > 2.61) & (pl.col("wc_reco_num_protons_35_MeV") > 0),
     r"WC NC $\Delta\rightarrow N\gamma$ $Np$ Selection"),
    ("wc_nc_delta_0p", (pl.col("wc_nc_delta_score") > 2.61) & (pl.col("wc_reco_num_protons_35_MeV") == 0),
     r"WC NC $\Delta\rightarrow N\gamma$ $0p$ Selection"),
    ("wc_inclusive_1g", erin_inclusive_1g_expr,
     r"WC Inclusive $1\gamma$ Selection"),
    ("wc_nueCC", pl.col("wc_nue_score") > 7,
     r"WC Inclusive $\nu_e$CC Selection"),
    ("wc_nueCC_Np", (pl.col("wc_nue_score") > 7) & (pl.col("wc_reco_num_protons_35_MeV") > 0),
     r"WC Inclusive $\nu_e$CC $Np$ Selection"),
    ("wc_nueCC_0p", (pl.col("wc_nue_score") > 7) & (pl.col("wc_reco_num_protons_35_MeV") == 0),
     r"WC Inclusive $\nu_e$CC $0p$ Selection"),
]

for name, expr, title in previously_existing_selections:
    sel_df = presel_merged_df.filter(expr)
    detvar_sel_df = detvar_presel_df.filter(expr) if detvar_presel_df is not None else None
    plot_sel_var(sel_df, "wc_kine_reco_Enu", r"WC Reconstructed $E_\nu$ (MeV)", np.linspace(0, 2500, 26),
                 "previously_existing_selections", f"{name}_reco_Enu", title,
                 detvar_sel_df=detvar_sel_df)

# BDT Scores For Every Topology

One 20-panel grid showing, for each topology, the distribution of that topology's multiclass BDT
probability over the full generic neutrino preselection (log $y$), comparing the runs 1--5 NuWro fake data to
the stacked prediction with a total prediction systematic band per panel.  Every panel uses the same
(preselection) event sample, so the reweighting covariance join is the same each time; only the plotted
score changes.

In [ ]:
BDT_SUBDIR = "bdt_scores_all_topologies"

def make_bdt_score_grid(df, name, weight_var="wc_net_weight_nuwro", include_data=True,
                        data_label="Runs 1-5 NuWro Fake Data"):
    """20-topology grid of multiclass BDT probability distributions over the full preselection
    (del1g_detailed breakdown, log y), with the NuWro fake data and a total prediction systematic band per panel."""
    assert_presel_df_intact()
    savename = sname(BDT_SUBDIR, name)

    pred_df = df.filter((pl.col("filetype") != "nuwro_fake_data") & pl.col(weight_var).is_not_null())
    data_df = df.filter(pl.col("filetype") == "nuwro_fake_data")
    pot = derive_normalizing_POT(pred_df, weight_var)
    mc_df = pred_df.filter(pl.col("filetype") != "ext")

    fig, axs = plt.subplots(7, 5, figsize=(20, 20))
    axs = axs.flatten()
    bins = np.linspace(0, 1, 21)
    bin_centers = (bins[:-1] + bins[1:]) / 2

    for i in tqdm(range(len(reco_categories))):
        var = f"prob_{reco_categories[i]}"
        signal_category_latex = reco_category_labels_latex[i]

        bottom = np.zeros(len(bins) - 1)
        for breakdown_label, breakdown_query, breakdown_color, breakdown_hatch, breakdown_label_latex in zip(
                del1g_detailed_category_labels, del1g_detailed_category_queries,
                del1g_detailed_category_colors, del1g_detailed_category_hatches,
                del1g_detailed_category_labels_latex):
            if "iso1g" in breakdown_label or "del1g" in breakdown_label:
                continue
            if breakdown_label in ["data", "nuwro_fake_data", "fullosc"]:
                continue
            curr_df = pred_df.filter(eval(breakdown_query, {"pl": pl, "__builtins__": {}}))
            counts = np.histogram(get_vals(curr_df, var), weights=get_vals(curr_df, weight_var), bins=bins)[0]
            axs[i].hist(bin_centers, weights=counts, bins=bins, bottom=bottom,
                        color=breakdown_color, hatch=breakdown_hatch, label=breakdown_label_latex)
            axs[i].hist(bin_centers, weights=counts, bins=bins, bottom=bottom,
                        histtype="step", color="k", lw=0.5)
            bottom = bottom + counts

        # total prediction systematic band (rw + detvar + prediction stat), as in make_reco_Enu_grid
        if use_rw_systematics:
            mc_counts = np.histogram(get_vals(mc_df, var), weights=get_vals(mc_df, weight_var), bins=bins)[0]
            rw_frac_cov_dic = get_rw_sys_frac_cov_matrices(
                mc_df, f"nwd_{BDT_SUBDIR}_{name}_{reco_categories[i]}", var, bins,
                dont_load_rw_from_systematic_cache=dont_load_systematics_from_cache,
                weights_df=weights_df, net_weight_var=weight_var)
            combined_frac_cov = np.zeros((len(bins) - 1, len(bins) - 1))
            for frac_cov in rw_frac_cov_dic.values():
                combined_frac_cov += frac_cov
            sys_cov = combined_frac_cov * np.outer(mc_counts, mc_counts)
            if use_detvar_systematics and detvar_presel_df is not None:
                detvar_frac_cov_dic = get_detvar_sys_frac_cov_matrices(
                    detvar_presel_df, f"nwd_{BDT_SUBDIR}_{name}_{reco_categories[i]}", var, bins,
                    dont_load_detvar_from_systematic_cache=dont_load_systematics_from_cache,
                    num_bootstrap_rounds_detvar=num_bootstrap_rounds_detvar,
                    num_bootstrap_samples_detvar=num_bootstrap_samples_detvar)
                for frac_cov in detvar_frac_cov_dic.values():
                    sys_cov += frac_cov * np.outer(mc_counts, mc_counts)
            sys_cov += get_pred_stat_cov(get_vals(pred_df, var), get_vals(pred_df, weight_var), bins)
            sys_errs = np.sqrt(np.diag(sys_cov))
            for bin_i in range(len(bins) - 1):
                # clamp the band bottom to stay drawable on the log axis
                band_bottom = max(bottom[bin_i] - sys_errs[bin_i], 1e-3)
                band_top = bottom[bin_i] + sys_errs[bin_i]
                rect = Rectangle((bins[bin_i], band_bottom),
                                 bins[bin_i + 1] - bins[bin_i], band_top - band_bottom,
                                 hatch=r"\\\\\\", fill=False, edgecolor="k", linewidth=0,
                                 label="Tot. Syst. Uncert." if bin_i == 0 else None)
                axs[i].add_patch(rect)

        max_y = np.max(bottom) if np.max(bottom) > 0 else 1
        if include_data:
            data_counts = np.histogram(get_vals(data_df, var), bins=bins)[0]
            axs[i].errorbar(bin_centers, data_counts, yerr=np.sqrt(data_counts), fmt="o", color="k", lw=0.5,
                            capsize=2, capthick=1, markersize=2, label=f"{pot:.2e} POT {data_label}")
            max_y = max(max_y, np.max(data_counts))

        axs[i].set_yscale("log")
        axs[i].set_ylim(0.1, max_y * 3)
        axs[i].set_xlim(0, 1)
        axs[i].set_title(signal_category_latex)
        # dashed guide line at the tuned BDT cut (the orthogonality cuts are not shown)
        if bdt_cut_thresholds[reco_categories[i]] is not None:
            axs[i].axvline(bdt_cut_thresholds[reco_categories[i]], color="k", ls="--", lw=1)
        if i == 19:
            axs[i].legend(ncol=4, loc="upper right", bbox_to_anchor=(-1, -0.5))
        if i in [15, 16, 17, 18, 19]:
            axs[i].set_xlabel("BDT Score")
        if i % 5 == 0: # only show y-label for leftmost column
            axs[i].set_ylabel(f"Counts (weighted\nto {pot:.2e} POT)")

    for axnum in range(len(reco_categories), len(axs)):
        axs[axnum].remove()

    fig.subplots_adjust(hspace=0.5, wspace=0.3, bottom=0.15)
    plt.savefig(f"../plots/{savename}.pdf")
    plt.savefig(f"../plots/{savename}.png")
    if show_plots:
        plt.show()
    else:
        plt.close(fig)

make_bdt_score_grid(presel_merged_df, "grid_bdt_scores")

# Dedicated BDT Score Distributions

Individual plots of the $1\gamma$ (`1gNp`, `1g0p`, `1gNp1mu`, `1g0p1mu`) and $\nu_e$CC (`nueCC_Np`, `nueCC_0p`)
multiclass BDT scores over the full generic neutrino preselection (log $y$), comparing runs 1-5 NuWro fake data to the
stacked prediction with the full systematic band, ratio, and $\chi^2$/$p$-value.  These are the same events,
variables, and binning as the corresponding grid panels above, so they reuse those covariance cache entries.
Saved under `bdt_scores_all_topologies/bdt_score_<cat>`.

In [ ]:
# dedicated BDT score plots for the 1gamma and nueCC topologies over the full generic preselection (log y);
# same events, variable, and binning as the grid panels above, so the selname is shared with them
for reco_category in ["1gNp", "1g0p", "1gNp1mu", "1g0p1mu", "nueCC_Np", "nueCC_0p"]:
    i = reco_categories.index(reco_category)
    plot_sel_var(presel_merged_df, f"prob_{reco_category}", f"{reco_category_labels_latex[i]} BDT Score",
                 np.linspace(0, 1, 21), BDT_SUBDIR, f"bdt_score_{reco_category}", "Generic Neutrino Preselection",
                 detvar_sel_df=detvar_presel_df, log_y=True, include_overflow=False,
                 # dashed guide line at the tuned BDT cut (the orthogonality cuts are not shown)
                 vertical_lines=None if bdt_cut_thresholds[reco_category] is None else [bdt_cut_thresholds[reco_category]],
                 selname=f"nwd_{BDT_SUBDIR}_grid_bdt_scores_{reco_category}")

# Reconstructed $E_\nu$ For Every Topology

20-topology grids of reconstructed $E_\nu$: the NuWro fake data (runs 1-5 combined) and the fake data split
by NuWro normalizing run period (runs 1, 2, 3, 4a, 4c, and 5; the run 4b/4d overlays are folded into the
run 4c group).  Individual per-topology plots are also made for use in the note.  A one-bin version of the
combined grid (each panel the total selected count, one bin spanning all reconstructed energies) is also made.
There is no "full prediction" grid here: that plot is not a fake-data comparison and lives in the open-data
notebook.

Each grid panel carries a total prediction systematic band (rw + detvar + prediction stat, mirroring
`make_histogram_plot`).  The combined grid shares its reweighting covariance cache entries with the
individual per-topology plots (same selname, binning, and bootstrap settings).  The detvar sample is a
fixed set of run 3b samples that cannot be split by run period, so the run-split grids reuse the combined
per-topology detvar fractional covariances.


In [ ]:
GRID_SUBDIR = "reco_Enu_all_topologies"

def make_reco_Enu_grid(df, name, weight_var="wc_net_weight_nuwro", include_data=True,
                       data_label="NuWro Fake Data", rw_selname_prefix=None, bins=None):
    """20-topology grid of stacked reco Enu predictions (del1g_detailed breakdown), optionally with data,
    with a total prediction systematic band (rw + detvar + prediction stat) per panel.
    bins defaults to the 20-bin 0-2000 MeV binning; pass a two-edge binning spanning all energies for a
    one-bin total-count grid (the covariance caches are keyed by binning as well as selname)."""
    assert_presel_df_intact()
    savename = sname(GRID_SUBDIR, name)
    if rw_selname_prefix is None:
        rw_selname_prefix = f"nwd_{GRID_SUBDIR}_{name}"

    pred_df = df.filter((pl.col("filetype") != "nuwro_fake_data") & pl.col(weight_var).is_not_null())
    data_df = df.filter(pl.col("filetype") == "nuwro_fake_data")
    pot = derive_normalizing_POT(pred_df, weight_var)

    fig, axs = plt.subplots(7, 5, figsize=(20, 20))
    axs = axs.flatten()
    if bins is None:
        bins = np.linspace(0, 2000, 21)
    bins = np.asarray(bins, dtype=float)
    bin_centers = (bins[:-1] + bins[1:]) / 2

    for i in tqdm(range(len(reco_categories))):
        signal_category_latex = reco_category_labels_latex[i]
        sel_pred_df = pred_df.filter(reco_category_queries[i])
        sel_data_df = data_df.filter(reco_category_queries[i])

        bottom = np.zeros(len(bins) - 1)
        for breakdown_label, breakdown_query, breakdown_color, breakdown_hatch, breakdown_label_latex in zip(
                del1g_detailed_category_labels, del1g_detailed_category_queries,
                del1g_detailed_category_colors, del1g_detailed_category_hatches,
                del1g_detailed_category_labels_latex):
            # the raw iso1g/del1g samples and the data-like categories are not part of the prediction
            if "iso1g" in breakdown_label or "del1g" in breakdown_label:
                continue
            if breakdown_label in ["data", "nuwro_fake_data", "fullosc"]:
                continue
            curr_df = sel_pred_df.filter(eval(breakdown_query, {"pl": pl, "__builtins__": {}}))
            counts = np.histogram(get_vals(curr_df, "wc_kine_reco_Enu"),
                                  weights=get_vals(curr_df, weight_var), bins=bins)[0]
            axs[i].hist(bin_centers, weights=counts, bins=bins, bottom=bottom,
                        color=breakdown_color, hatch=breakdown_hatch, label=breakdown_label_latex)
            axs[i].hist(bin_centers, weights=counts, bins=bins, bottom=bottom,
                        histtype="step", color="k", lw=0.5)
            bottom = bottom + counts

        # total prediction systematic band (rw + detvar + prediction stat), mirroring make_histogram_plot:
        # the fractional covariances scale the non-EXT MC prediction, drawn around the total prediction
        if use_rw_systematics:
            mc_df = sel_pred_df.filter(pl.col("filetype") != "ext")
            mc_counts = np.histogram(get_vals(mc_df, "wc_kine_reco_Enu"),
                                     weights=get_vals(mc_df, weight_var), bins=bins)[0]
            rw_frac_cov_dic = get_rw_sys_frac_cov_matrices(
                mc_df, f"{rw_selname_prefix}_{reco_categories[i]}", "wc_kine_reco_Enu", bins,
                dont_load_rw_from_systematic_cache=dont_load_systematics_from_cache,
                weights_df=weights_df, net_weight_var=weight_var)
            combined_frac_cov = np.zeros((len(bins) - 1, len(bins) - 1))
            for frac_cov in rw_frac_cov_dic.values():
                combined_frac_cov += frac_cov
            sys_cov = combined_frac_cov * np.outer(mc_counts, mc_counts)
            if use_detvar_systematics and detvar_presel_df is not None:
                # shared selname with the individual per-topology Enu plots (same binning and
                # bootstrap settings), so the detvar bootstrap is only done once per topology
                detvar_cat_df = detvar_presel_df.filter(reco_category_queries[i])
                detvar_frac_cov_dic = get_detvar_sys_frac_cov_matrices(
                    detvar_cat_df, f"nwd_{GRID_SUBDIR}_reco_Enu_{reco_categories[i]}", "wc_kine_reco_Enu", bins,
                    dont_load_detvar_from_systematic_cache=dont_load_systematics_from_cache,
                    num_bootstrap_rounds_detvar=num_bootstrap_rounds_detvar,
                    num_bootstrap_samples_detvar=num_bootstrap_samples_detvar)
                for frac_cov in detvar_frac_cov_dic.values():
                    sys_cov += frac_cov * np.outer(mc_counts, mc_counts)
            sys_cov += get_pred_stat_cov(get_vals(sel_pred_df, "wc_kine_reco_Enu"),
                                         get_vals(sel_pred_df, weight_var), bins)
            sys_errs = np.sqrt(np.diag(sys_cov))
            for bin_i in range(len(bins) - 1):
                rect = Rectangle((bins[bin_i], bottom[bin_i] - sys_errs[bin_i]),
                                 bins[bin_i + 1] - bins[bin_i], 2 * sys_errs[bin_i],
                                 hatch=r"\\\\\\", fill=False, edgecolor="k", linewidth=0,
                                 label="Tot. Syst. Uncert." if bin_i == 0 else None)
                axs[i].add_patch(rect)

        max_y = np.max(bottom) if np.max(bottom) > 0 else 1
        if include_data:
            data_counts = np.histogram(get_vals(sel_data_df, "wc_kine_reco_Enu"), bins=bins)[0]
            axs[i].errorbar(bin_centers, data_counts, yerr=np.sqrt(data_counts), fmt="o", color="k", lw=0.5,
                            capsize=2, capthick=1, markersize=2, label=f"{pot:.2e} POT {data_label}")
            max_y = max(max_y, np.max(data_counts))

        axs[i].set_ylim(0, max_y * 1.1)
        axs[i].set_xlim(bins[0], bins[-1])
        if len(bins) == 2:  # one-bin total-count grid: the x axis carries no information
            axs[i].set_xticks([])
        axs[i].set_title(f"{signal_category_latex} Selection")
        if i == 19:
            axs[i].legend(ncol=4, loc="upper right", bbox_to_anchor=(-1, -0.5))
        if i in [15, 16, 17, 18, 19]:
            axs[i].set_xlabel(r"WC Reconstructed $E_\nu$ (MeV)")
        if i % 5 == 0: # only show y-label for leftmost column
            axs[i].set_ylabel(f"Counts (weighted\nto {pot:.2e} POT)")

    for axnum in range(len(reco_categories), len(axs)):
        axs[axnum].remove()

    fig.subplots_adjust(hspace=0.5, wspace=0.3, bottom=0.15)
    plt.savefig(f"../plots/{savename}.pdf")
    plt.savefig(f"../plots/{savename}.png")
    if show_plots:
        plt.show()
    else:
        plt.close(fig)

In [ ]:
# NuWro fake data, runs 1-5 combined (shares its rw covariance cache with the individual per-topology plots)
make_reco_Enu_grid(presel_merged_df, "grid_nuwro", include_data=True, data_label="Runs 1-5 NuWro Fake Data",
                   rw_selname_prefix=f"nwd_{GRID_SUBDIR}_reco_Enu")

In [ ]:
# one-bin version of the combined grid: each panel is the total selected count, with a single reco Enu
# bin spanning all energies (0-999 GeV, so nothing overflows), comparing runs 1-5 NuWro fake data to the prediction with the full
# systematic band; the distinct name and binning keep its covariance cache entries separate
make_reco_Enu_grid(presel_merged_df, "grid_nuwro_one_bin", include_data=True, data_label="Runs 1-5 NuWro Fake Data",
                   bins=np.array([0.0, 999_000.0]))

In [ ]:
# NuWro fake data split by normalizing run period (the NuWro config's groups):
# runs 1, 2, 3, 4a and 5 map to themselves; runs 4b/4c/4d/4bcd overlays -> run 4c NuWro
nuwro_groups = {"1": "Run 1", "2": "Run 2", "3": "Run 3", "4a": "Run 4a", "4c": "Run 4c", "5": "Run 5"}
for group, label in nuwro_groups.items():
    sub = presel_merged_df.filter(pl.col("normalizing_run_period_nuwro") == group)
    make_reco_Enu_grid(sub, f"grid_nuwro_run{group}", include_data=True, data_label=f"{label} NuWro Fake Data")

In [ ]:
# individual per-topology NuWro-fake-data reco Enu plots (for the note)
for i, reco_category in enumerate(reco_categories):
    sel_df = presel_merged_df.filter(reco_category_queries[i])
    detvar_cat_df = detvar_presel_df.filter(reco_category_queries[i]) if detvar_presel_df is not None else None
    plot_sel_var(sel_df, "wc_kine_reco_Enu", r"WC Reconstructed $E_\nu$ (MeV)", np.linspace(0, 2000, 21),
                 GRID_SUBDIR, f"reco_Enu_{reco_category}", f"{reco_category_labels_latex[i]} Selection",
                 detvar_sel_df=detvar_cat_df)

# One-Bin Selections

Individual one-bin plots for the four in-FV $1\gamma$ selections (`1gNp`, `1g0p`, `1gNp1mu`, `1g0p1mu`) and the
two $\nu_e$CC selections (`nueCC_Np`, `nueCC_0p`): a single reconstructed $E_\nu$ bin spanning all energies
(0-999 GeV, with the overflow bin turned off; an always-empty 999-2000 GeV bin is drawn beside it only to
make room for the legend), so each plot is the total selected event count comparing
runs 1-5 NuWro fake data to the stacked prediction with the full systematic band, ratio, and $\chi^2$/$p$-value.
Each plot also gets its systematic breakdown plot (fractional uncertainty per source, saved alongside as
`<name>_systematic_breakdown.pdf`, with the numbers printed).  Saved under `one_bin_selections/`.

The two $1\gamma 1\mu$ selections are also plotted with the $\nu_\mu$CC radiative-correction prediction removed
(`one_bin_<cat>_no_rad_corr`), to see how sensitive the data comparison is to that process.

In [ ]:
ONE_BIN_SUBDIR = "one_bin_selections"
# one reco Enu bin spanning all energies (0-999 GeV) plus an always-empty 999-2000 GeV bin, which only
# makes room for the legend (make_histogram_plot's own overflow bin is turned off)
one_bin_bins = np.array([0.0, 999_000.0, 2_000_000.0])

for reco_category in ["1gNp", "1g0p", "1gNp1mu", "1g0p1mu", "nueCC_Np", "nueCC_0p"]:
    i = reco_categories.index(reco_category)
    sel_df = presel_merged_df.filter(reco_category_queries[i])
    detvar_cat_df = detvar_presel_df.filter(reco_category_queries[i]) if detvar_presel_df is not None else None
    plot_sel_var(sel_df, "wc_kine_reco_Enu", r"WC Reconstructed $E_\nu$ (MeV), single bin", one_bin_bins,
                 ONE_BIN_SUBDIR, f"one_bin_{reco_category}", f"{reco_category_labels_latex[i]} Selection",
                 detvar_sel_df=detvar_cat_df, include_overflow=False,
                 # systematic breakdown plot (requires both systematics toggles) with the numbers printed
                 plot_sys_breakdown=use_rw_systematics and use_detvar_systematics, print_sys_breakdown=True)

# alternate versions of the 1gamma 1mu one-bin plots without the numuCC radiative-correction prediction,
# to see how sensitive the data comparison is to that process
for reco_category in ["1gNp1mu", "1g0p1mu"]:
    i = reco_categories.index(reco_category)
    sel_df = presel_merged_df.filter(reco_category_queries[i] & (pl.col("filetype") != "numuCC_rad_corrected"))
    detvar_cat_df = detvar_presel_df.filter(reco_category_queries[i]) if detvar_presel_df is not None else None
    plot_sel_var(sel_df, "wc_kine_reco_Enu", r"WC Reconstructed $E_\nu$ (MeV), single bin", one_bin_bins,
                 ONE_BIN_SUBDIR, f"one_bin_{reco_category}_no_rad_corr",
                 f"{reco_category_labels_latex[i]} Selection, no $\\nu_\\mu$CC rad. corr. prediction",
                 detvar_sel_df=detvar_cat_df, include_overflow=False,
                 plot_sys_breakdown=use_rw_systematics and use_detvar_systematics, print_sys_breakdown=True)

# $1\gamma$ Selections

Separate plots for NC and $\nu_\mu$CC, $Np$ and $0p$: `1gNp`, `1g0p`, `1gNp1mu`, `1g0p1mu`.
Photon kinematics for all four; leading proton energy/angle for the $Np$ selections and proton
multiplicity for all; muon kinematics for the $1\mu$ selections; position-in-TPC variables for all.


In [ ]:
one_g_categories = ["1gNp", "1g0p", "1gNp1mu", "1g0p1mu"]

for cat in one_g_categories:
    vars_here = list(photon_vars)
    vars_here.append(proton_multiplicity_var)
    if "Np" in cat:
        vars_here += leading_proton_vars + summed_proton_vars
    if "1mu" in cat:
        vars_here += muon_vars
        vars_here.append(("reco_muon_photon_opening_angle", r"Reco Muon-Photon Opening Angle (deg)",
                          np.linspace(0, 180, 19), "muon_photon_opening_angle"))
    vars_here += position_vars
    plot_category_vars(cat, "1g_selections", vars_here)

# $2\gamma$ Selections

Separate plots for NC and $\nu_\mu$CC, $Np$ and $0p$, out-FV, and the higher-mass-resonance ($\eta$)
selection: `NC1pi0_Np`, `NC1pi0_0p`, `numuCC1pi0_Np`, `numuCC1pi0_0p`, `1pi0_outFV`, `eta_other`.
Leading/subleading photon kinematics and the two-photon system for all; proton and muon kinematics where
applicable.  The invariant mass binning extends to 1000 MeV for the $\eta$ selection.

In [ ]:
two_g_categories = ["NC1pi0_Np", "NC1pi0_0p", "numuCC1pi0_Np", "numuCC1pi0_0p", "1pi0_outFV", "eta_other"]

for cat in two_g_categories:
    mass_bins = np.linspace(0, 1000, 41) if cat == "eta_other" else np.linspace(0, 500, 26)
    vars_here = list(two_photon_leading_vars) + two_photon_system_vars(mass_bins)
    if cat not in ["1pi0_outFV", "eta_other"]:
        vars_here.append(proton_multiplicity_var)
    if "Np" in cat:
        vars_here += leading_proton_vars + summed_proton_vars
    if "numuCC" in cat:
        vars_here += muon_vars
    plot_category_vars(cat, "2g_selections", vars_here)

# $\nu_e$CC Selections

`nueCC_Np` and `nueCC_0p`.  The WC primary shower is the electron candidate.  The
electron-leading-proton opening angle is only made for the $Np$ selection.

In [ ]:
for cat in ["nueCC_Np", "nueCC_0p"]:
    vars_here = [
        ("wc_reco_showerKE", "Reco Electron Energy (GeV)", np.linspace(0, 2.5, 26), "electron_energy"),
        ("wc_reco_shower_theta", r"Reco Electron $\theta$ (deg)", np.linspace(0, 180, 19), "electron_theta"),
        proton_multiplicity_var,
    ]
    if "Np" in cat:
        vars_here += leading_proton_vars + summed_proton_vars
        vars_here.append(("reco_shower_proton_opening_angle", "Reco Electron-Leading-Proton Opening Angle (deg)",
                          np.linspace(0, 180, 19), "electron_proton_opening_angle"))
    plot_category_vars(cat, "nueCC_selections", vars_here)

# $\nu_\mu$CC Selections

`numuCC_Np` and `numuCC_0p`.  The muon-leading-proton opening angle is only made for the $Np$ selection.

In [ ]:
for cat in ["numuCC_Np", "numuCC_0p"]:
    vars_here = list(muon_vars)
    vars_here.append(proton_multiplicity_var)
    if "Np" in cat:
        vars_here += leading_proton_vars + summed_proton_vars
        vars_here.append(("reco_muon_proton_opening_angle", "Reco Muon-Leading-Proton Opening Angle (deg)",
                          np.linspace(0, 180, 19), "muon_proton_opening_angle"))
    plot_category_vars(cat, "numuCC_selections", vars_here)

# NC $0\gamma$ Selection

Proton kinematics for the `NC_no_gamma` selection.

In [ ]:
plot_category_vars("NC_no_gamma", "NC0g_selection",
                   [proton_multiplicity_var] + leading_proton_vars + summed_proton_vars)

# Other Selections

Multi-$\pi^0$ and $\pi^0$ Dalitz decay.  The "combined" energies use the summed WC KINE two-photon
candidate energies; the shower multiplicity is the number of > 20 MeV reco showers.

In [ ]:
shower_multiplicity_var = ("wc_shw_sp_n_20mev_showers", "Reco Number of > 20 MeV Showers",
                           np.arange(-0.5, 8.5, 1.0), "photon_multiplicity")

plot_category_vars("multi_pi0", "other_selections", [
    shower_multiplicity_var,
    ("wc_kine_pio_energy_1", "Reco Leading Photon Energy (MeV)", np.linspace(0, 800, 21), "leading_photon_energy"),
    ("reco_two_photon_energy_sum", "Reco Combined Photon Energy (MeV)", np.linspace(0, 1500, 31), "combined_photon_energy"),
])

plot_category_vars("pi0_dalitz_decay", "other_selections", [
    shower_multiplicity_var,
    ("wc_reco_showerKE", "Reco Leading Shower Energy (GeV)", np.linspace(0, 1.5, 16), "leading_shower_energy"),
    ("reco_two_photon_energy_sum", "Reco Combined Shower Energy (MeV)", np.linspace(0, 1500, 31), "combined_shower_energy"),
])